# Módulo 7 — Cointegração
## Seção 7.5 — Fundamentos de VAR e teste de Johansen: posto de cointegração

**Estado:** material didático para revisão científica. **Pré-requisitos:** §§7.1–7.4; álgebra linear (autovalores, posto, matrizes); regressão multivariada, $I(1)$. **Objetivo:** entender por que mais de duas séries podem ter diversas relações cointegrantes, por que isso exige uma abordagem de sistemas e como interpretar os testes de Johansen.

**Controle bibliográfico:** o **Volume 1, Modelos Lineares Univariados**, de Morettin e Toloi (2018) oferece pré-requisitos em processos integrados, raízes unitárias e dinâmica ARIMA (cap. 5; Apêndice B); **não é referência direta** para o teste de Johansen. Esta seção é complementação fundamentada na teoria de sistemas cointegrados de Johansen (1988, 1991) e na documentação técnica da implementação do Statsmodels. A derivação integral por máxima verossimilhança do problema de autovalores generalizados não é afirmada aqui; será assunto de aprofundamento específico.

### 1. Da regressão entre duas séries ao vetor de variáveis

Com $p$ séries, escrevemos $\mathbf y_t=(y_{1t},\dots,y_{pt})^\top$. Um modelo autorregressivo vetorial de ordem $k$ — VAR($k$) — é

$$\boxed{\mathbf y_t=\mathbf c+A_1\mathbf y_{t-1}+\cdots+A_k\mathbf y_{t-k}+\boldsymbol\varepsilon_t,}$$

onde $A_i$ são matrizes $p\times p$; $\mathbf c$ é um vetor; $\boldsymbol\varepsilon_t$ tem média zero e matriz de covariância $\Sigma$ positiva definida sob as hipóteses usuais. Cada equação pode depender das defasagens de **todas** as séries. Um VAR em níveis com componentes $I(1)$ não deve ser interpretado automaticamente usando a inferência estacionária padrão.

### 2. Dedução algébrica da representação de correção de erros

Comece pelo VAR(2), sem intercepto para destacar a álgebra:

$$\mathbf y_t=A_1\mathbf y_{t-1}+A_2\mathbf y_{t-2}+\boldsymbol\varepsilon_t.$$

Subtraia $\mathbf y_{t-1}$ de ambos os lados e use $\mathbf y_{t-2}=\mathbf y_{t-1}-\Delta\mathbf y_{t-1}$:

$$\Delta\mathbf y_t=(A_1+A_2-I)\mathbf y_{t-1}-A_2\Delta\mathbf y_{t-1}+\boldsymbol\varepsilon_t.$$

Defina $\Pi=A_1+A_2-I$ e $\Gamma_1=-A_2$; logo

$$\boxed{\Delta\mathbf y_t=\Pi\mathbf y_{t-1}+\Gamma_1\Delta\mathbf y_{t-1}+\boldsymbol\varepsilon_t.}$$

Para VAR($k$), o mesmo procedimento leva a

$$\Delta\mathbf y_t=\Pi\mathbf y_{t-1}+\sum_{j=1}^{k-1}\Gamma_j\Delta\mathbf y_{t-j}+\mathbf d_t+\boldsymbol\varepsilon_t,$$

com $\Pi=\sum_{i=1}^k A_i-I$ e $\Gamma_j=-\sum_{i=j+1}^{k} A_i$. **A passagem VAR → representação em diferenças é uma identidade algébrica**, não é por si só prova de cointegração.

### 3. Posto da matriz de longo prazo

Sob hipóteses apropriadas sobre raízes e regularidade para um sistema $I(1)$:

- $\operatorname{rank}(\Pi)=0$: $\Pi=0$; não existe termo linear de correção de erros por esta representação.
- $0<r=\operatorname{rank}(\Pi)<p$: existem $r$ relações cointegrantes linearmente independentes e podemos escrever $\Pi=\alpha\beta^\top$, com $\alpha,\beta$ matrizes $p\times r$ de posto coluna completo.
- $r=p$: $\Pi$ tem posto completo; no caso regular usual, o sistema não é um sistema $I(1)$ cointegrado com tendências comuns da forma anterior, mas apresenta comportamento estacionário em níveis (dependendo dos demais autovalores do sistema).

As colunas de $\beta$ representam as **combinações estacionárias**; $\alpha$ governa como cada componente responde aos desvios. A fatoração não é única: $\alpha\beta^\top=(\alpha H)(\beta H^{-\top})^\top$ para qualquer matriz invertível $H\in\mathbb R^{r\times r}$ (equivalentemente, escolha normalizações dos vetores).

**Exemplo matricial.** Com $p=2$, $\alpha=(-0{,}3,\,0{,}1)^\top$ e $\beta=(1,-2)^\top$:

$$\Pi=\alpha\beta^\top=\begin{pmatrix}-0{,}3&0{,}6\\0{,}1&-0{,}2\end{pmatrix},\qquad \det(\Pi)=0,\quad \operatorname{rank}(\Pi)=1.$$

Então o termo $\Pi\mathbf y_{t-1}$ depende apenas da combinação $y_{1,t-1}-2y_{2,t-1}$. Os dois componentes podem reagir ao mesmo desvio com intensidades e sinais distintos.

### 4. Como Johansen infere o posto?

O método baseado em máxima verossimilhança reduzida em posto, sob uma especificação VAR gaussiana apropriada e pressupostos de integração, constrói autovalores estimados $\widehat\lambda_1\ge\cdots\ge\widehat\lambda_p\ge 0$, associados à informação de longo prazo **após controlar a dinâmica de curto prazo e os determinísticos**.

Para $r_0\in\{0,\ldots,p-1\}$, a estatística do traço é

$$\boxed{J_{trace}(r_0)=-T_{eff}\sum_{i=r_0+1}^{p}\log(1-\widehat\lambda_i).}$$

Testa $H_0:\operatorname{rank}(\Pi)\le r_0$ contra posto maior. Já a estatística do máximo autovalor é

$$\boxed{J_{max}(r_0,r_0+1)=-T_{eff}\log(1-\widehat\lambda_{r_0+1}),}$$

que contrasta posto $r_0$ com posto $r_0+1$ na formulação sequencial usual. $T_{eff}$ é o tamanho efetivo após defasagens e ajustes, **não necessariamente o número bruto de linhas**. As distribuições sob a hipótese nula são **não usuais** e dependem dos termos determinísticos; não usar quantis de $\chi^2$ convencional.

**Escolha prévia crucial:** tendência restrita/irrestrita, constante, defasagens em diferenças, frequência de amostragem e possíveis quebras. Na API `coint_johansen(endog, det_order, k_ar_diff)`, `det_order=-1/0/1` seleciona, respectivamente, ausência de determinístico, constante ou tendência linear na parametrização limitada dessa rotina; **essas opções não abrangem todas as cinco especificações determinísticas de Johansen**. `k_ar_diff` indica a quantidade de defasagens das diferenças, não a ordem do VAR em níveis: em uma representação VAR($k$), frequentemente `k_ar_diff=k-1`.

### 5. Experimento com duas séries e um vetor cointegrante

Construímos $X_t$ passeio aleatório e $Y_t=1{,}8X_t+u_t$, onde $u_t$ é AR(1) estacionário. Existe exatamente uma combinação estacionária por construção: $Y_t-1{,}8X_t=u_t$; a outra direção contém a tendência estocástica comum.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.vector_ar.vecm import coint_johansen

rng = np.random.default_rng(755)
n=650
x=np.cumsum(rng.normal(size=n))
u=np.zeros(n)
noise=rng.normal(scale=0.6,size=n)
for t in range(1,n):
    u[t]=0.5*u[t-1]+noise[t]
y=1.8*x+u
Z=np.column_stack([y,x])
res=coint_johansen(Z,det_order=0,k_ar_diff=1)
print('Autovalores estimados:',np.round(res.eig,4))
print('Traço:',np.round(res.lr1,3))
print('Críticos de traço [90%,95%,99%] por hipótese de posto:')
print(np.round(res.cvt,3))
print('Máximo autovalor:',np.round(res.lr2,3))
print('Críticos do máximo autovalor:')
print(np.round(res.cvm,3))
# A primeira coluna é um autovetor estimado; normalizamos seu coeficiente em Y para 1.
beta_est = res.evec[:,0]/res.evec[0,0]
print('Vetor normalizado estimado [Y,X]:',np.round(beta_est,3))

fig,axes=plt.subplots(2,1,figsize=(9,5.5),sharex=True)
axes[0].plot(y,label='Y');axes[0].plot(x,label='X');axes[0].legend()
axes[1].plot(y-1.8*x,label='Y − 1.8 X (relação verdadeira)')
axes[1].plot(Z@beta_est,label='Relação estimada',alpha=0.65)
axes[1].legend();axes[1].set_xlabel('Tempo')
for ax in axes:ax.grid(alpha=0.2)
fig.tight_layout();plt.savefig('figuras/07_05_johansen.png',dpi=145);plt.show()

**Figura 7.5.1 —** Níveis integrados e combinação cointegrante verdadeira e estimada. A normalização do autovetor permite comparação; o sinal global do autovetor é arbitrário.

**Interpretação passo a passo:** compare `res.lr1[0]` ao valor crítico `res.cvt[0,1]` de 95% para testar posto no máximo zero; se rejeitado, examine `res.lr1[1]` contra `res.cvt[1,1]` para posto no máximo um. Na simulação geradora esperamos $r=1$, mas resultados finitos variam. A escolha dos determinísticos na rotina **não é garantia** de que a especificação seja a mais adequada para dados reais.

### 6. Limitações e conexão com o próximo capítulo

Johansen não transforma uma relação estatística em mecanismo causal. Quebras estruturais, tendência determinística mal especificada, resíduos autocorrelacionados, heterocedasticidade, dimensões altas e amostras pequenas podem distorcer resultados. **Não** misture indiscriminadamente séries $I(0)$ e $I(1)$ ou séries $I(2)$ sem reformular a teoria. Em um VECM completo, as matrizes $\alpha$, $\beta$ e $\Gamma_j$ são estimadas conjuntamente sob restrições de posto; isso será feito na Seção 7.6, que também abordará verificação de estabilidade e previsão.

### 7. Exercícios guiados

**A.** Desenvolva toda a álgebra da transformação VAR(3) → diferenças e obtenha $\Pi$, $\Gamma_1$, $\Gamma_2$.

**B.** Verifique o posto da matriz numérica $\Pi$ pelo determinante e pelas linhas proporcionais. Identifique $\alpha$ e $\beta$ e discuta normalização.

**C.** Antes de executar, elabore pseudocódigo para comparar `lr1` com `cvt` sequencialmente; repita a simulação para sementes e tamanhos distintos.

**D.** Se $p=3$ e $r=2$, quantas tendências estocásticas comuns espera-se na formulação $I(1)$ regular? Interprete $p-r$.

**E.** Explique por que interpretar `k_ar_diff=2` como VAR(2) em níveis está errado na parametrização usual.

### Referências

**Livro principal (pré-requisitos, não Johansen):** Morettin; Toloi (2018), cap. 5 e Apêndice B. **Fontes específicas complementares:** Johansen, S. (1988), *Statistical analysis of cointegration vectors*, *Journal of Economic Dynamics and Control*, 12(2–3), 231–254; Johansen, S. (1991), *Estimation and hypothesis testing of cointegration vectors in Gaussian vector autoregressive models*, *Econometrica*, 59(6), 1551–1580. **Referência operacional:** documentação do Statsmodels `coint_johansen` (versão instalada 0.14.6). As passagens algébricas VAR(2) e o exemplo matricial são deduções didáticas explícitas.